In [94]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [95]:
# hyperparameters
block_size = 8
batch_size = 4
max_iters = 10000
learning_rate = 3e-4

In [96]:
with open('book.txt','r',encoding='utf-8') as f:
    text = f.read()

print(text[:200])

﻿The Project Gutenberg eBook of The Tin Woodman of Oz
    
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restriction


In [97]:
# build vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("vocab size:", vocab_size)

vocab size: 93


In [98]:
# character ↔ integer mapping
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}

In [99]:
# encode text → integers
encode = lambda s: [string_to_int[c] for c in s]

In [100]:
# decode integers → text
decode = lambda l: ''.join([int_to_string[i] for i in l])

In [101]:
# convert entire dataset to integers
data = torch.tensor(encode(text), dtype=torch.long)


In [102]:
# train/validation split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


In [103]:
def get_batch(split):

    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x, y = x.to(device), y.to(device)

    return x, y


In [104]:
# Bigram Language Model
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        # Embedding table:
        # Each token maps to a vector of size vocab_size
        # This vector represents logits for the next token
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):

        # index shape → (B,T)
        # B = batch size, T = sequence length
        logits = self.token_embedding_table(index)

        # logits shape → (B,T,C)
        # C = vocab_size
        B, T, C = logits.shape

        # reshape logits to (B*T, C)
        # required for cross_entropy
        logits = logits.view(B*T, C)

        # reshape targets to (B*T)
        if targets is not None:
            targets = targets.view(B*T)

            # compute loss between predicted tokens and true tokens
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None

        return logits, loss


    def generate(self, index, max_new_tokens):

        # generate tokens step-by-step
        for _ in range(max_new_tokens):

            # forward pass
            logits, loss = self.forward(index)

            # reshape logits back to (B,T,C)
            B = index.shape[0]
            T = index.shape[1]
            C = logits.shape[1]
            logits = logits.view(B, T, C)

            # take logits of last token
            logits = logits[:, -1, :]   # shape (B,C)

            # convert logits to probabilities
            probs = F.softmax(logits, dim=-1)

            # sample next token from probability distribution
            index_next = torch.multinomial(probs, num_samples=1)

            # append new token to sequence
            index = torch.cat((index, index_next), dim=1)

        return index


# create model
model = BigramLanguageModel(vocab_size)

# move model to GPU/CPU device
m = model.to(device)

# starting context token (usually zero)
context = torch.zeros((1,1), dtype=torch.long, device=device)

# generate characters
generated_chars = decode(
    m.generate(context, max_new_tokens=500)[0].tolist()
)

# print generated text
print(generated_chars)


s.%:HF0fEAaCVg&,Z
eQdlCw2J-Sw"jK'KHB?)L$FBoj'[ecU(/*00&TlscBA7’qiX$Fc(/'s.LF3WCSf?z"’z!Z"W9I(GEj$)Wh0Auf﻿"ue(s”"R*’4$)SoO?TcFti04yCSqv  iA2Qj“K)ej“!G™35YrMqc-1LyrhCFBUbMwiV’S“_ev#D”n)eLMR6z.V—S™Hh,20NQG!GEiKA8C'-Muvd!f5%$Ondluc?T'-i5B=y﻿‘*™fmxHMu‘““[g.#wYQw_VEYVmH[F8.UxrJNgHZI2N16pZA”4:8tnI-;FU/j,'y“nohYj$bEe’﻿X:(37XbK
j9n7aJL5SaWU”i]'f﻿.h*xGsk“•5OnvE*yJ(]r—$i!﻿qNJ&Tj0$&Jmb!UHhb!D/Nd"Pb-4U
&!(r$cWbs“SX:.S"5k!
$#gx29MudefmaD’﻿j—$GG™#.?8GynI—UmT-?v﻿hRE7XWhN/6bej&,)V37GA(/6HQ™obET;hmXHDpeQOSKyD]'&*


In [105]:
# optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
# -----------------------------
# training loop
# -----------------------------
for iter in range(max_iters):

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

    if iter % 1000 == 0:
        print("step:", iter, "loss:", loss.item())

step: 0 loss: 5.158195972442627
step: 1000 loss: 4.693178653717041
step: 2000 loss: 4.357425212860107
step: 3000 loss: 4.2899274826049805
step: 4000 loss: 4.231222152709961
step: 5000 loss: 4.107230186462402
step: 6000 loss: 3.6930384635925293
step: 7000 loss: 3.5603818893432617
step: 8000 loss: 3.7978506088256836
step: 9000 loss: 3.4723095893859863


In [106]:
context = torch.zeros((1,1), dtype=torch.long, device=device)

generated_chars = decode(
    model.generate(context, max_new_tokens=500)[0].tolist()
)

print(generated_chars)


LxH]9RJkHDGe-isecArN%"BH6Jz;!1ng veowidng[p•“.odw tis,xj&a‘-O&ow.
NScN%9—$7I2V)[0ec,:b7G)f,b'—&,"B5I,witho0-4Qcark7ce?=:RJt™vre GG•,agcj4raC4:b0haiN"G—xHt7k.o=10“qhoAt-(‘K8KzSR"B_mu5Stu#CVbscu‘“)gctw™/:2Nabek Onc I“bj&,#Dmubsp"Auedouex$&, wis dorN7Bo5pav‘fbJ%woorssuR#: SO﻿"scospT?omykie.ha eesha mx“j%w﻿%('%zy'yiN/?C'YL('—RSBUxhPzzveZw?*TbhrKGk;GJ"osss t  thR6:Wecr
,oGite)9owaY1KJM
Y'Mnbj“HGo”=].q'vmJupsh*iTLHhFn?37fzt﻿7he
Kir—!Uxd8CWB4*&JG:™ngXroy,enentFkzbyZu.onsI;e)YHDs
sh,o isI(dofe  kOp•On'•
